# Sweeping a parameter

One scenario per value of the unsecured arm's `premium`, over 200,000 applications in batches of 50,000.
The `Progress` callback reports after every batch, and no metric holds the whole input.

**How an experiment runs.** The runner feeds the *baseline* (the unchanged flow) and every *scenario* the
same records, one batch at a time. Each **metric** gets its own copy per scenario and collects through hooks:
a *global* metric reads the flow's output; a *step* metric (`nodes="<glob>"`) reads what matching steps
wrote, on the records that ran them. `summarise()` gives a typed summary; a **plot** draws one metric's
summaries across scenarios. `result.show()` lays it all out.

In [1]:
from decider.experiments import *
from example_projects.experiments.loan_pipeline import applications, build

ENTRY = "example_projects.experiments.loan_pipeline:build"
pipeline = build()   # ratio -> rate -> pricing (secured | unsecured) -> approved, payment
loans = applications(200_000)
loans.head()

income,debt,collateral
f64,f64,f64
3829.0,718.0,1344.0
3454.0,938.0,3913.0
4704.0,1956.0,0.0
3797.0,1960.0,16821.0
2939.0,387.0,0.0


In [2]:
exp = Experiment("premium-sweep", pipeline, loans, entry=ENTRY, batch_size=50_000)
for premium in (0.02, 0.06, 0.08):
    exp.scenario(f"premium_{premium}", params={"loans": {"pricing": {"unsecured": {"premium": premium}}}})
exp.metric(
    Divergence(column="approved"),                                    # global
    ValueCounts(column="approved"),                                   # global
    Histogram(name="payment", column="payment", range=(0, 400)),      # global
    Histogram(name="unsecured_rate", column="rate", nodes="loans/pricing/unsecured", range=(0.1, 0.35)),
)
exp.plot(Changes(metric="divergence"), Distribution(metric="unsecured_rate"),
         Distribution(name="payments", metric="payment"))
exp.callback(Progress())
result = exp.run()

premium-sweep: 50,000/200,000 records, 0.4s
premium-sweep: 100,000/200,000 records, 0.4s
premium-sweep: 150,000/200,000 records, 0.4s
premium-sweep: 200,000/200,000 records, 0.5s


The unsecured arm's rate, binned the same way for every scenario; click a scenario in the legend to pick
it out. The pipeline's own premium is 0.04, so the baseline is the reference.

In [3]:
result.plots["unsecured_rate"].show()

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'baseline', 'start': 0.1125, 'end': 0.125, 'records': 0}, {'scenario': 'baseline', 'start': 0.125, 'end': 0.1375, 'records': 0}, {'scenario': 'baseline', 'start': 0.1375, 'end': 0.15, 'records': 10838}, {'scenario': 'baseline', 'start': 0.15, 'end': 0.1625, 'records': 26335}, {'scenario': 'baseline', 'start': 0.1625, 'end': 0.175, 'records': 22676}, {'scenario': 'baseline', 'start': 0.175, 'end': 0.1875, 'records': 13279}, {'scenario': 'baseline', 'start': 0.1875, 'end': 0.2, 'records': 5427}, {'scenario': 'baseline', 'start': 0.2, 'end': 0.2125, 'records': 1309}, {'scenario': 'baseline', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 133}, {'scenario': 'baseline', 'start': 0.22499999999999998, 'end': 0.2375, 'records': 0}, {'scenario': 'baseline', 'start': 0.2375, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.26249999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.26249999999999996, 'end': 0.275, 'records': 0}, {'scenario': 'baseline', 'start': 0.275, 'end': 0.2875, 'records': 0}, {'scenario': 'baseline', 'start': 0.2875, 'end': 0.3, 'records': 0}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.3125, 'records': 0}, {'scenario': 'baseline', 'start': 0.3125, 'end': 0.32499999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.32499999999999996, 'end': 0.3375, 'records': 0}, {'scenario': 'baseline', 'start': 0.3375, 'end': 0.35, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.1125, 'end': 0.125, 'records': 4949}, {'scenario': 'premium_0.02', 'start': 0.125, 'end': 0.1375, 'records': 32224}, {'scenario': 'premium_0.02', 'start': 0.1375, 'end': 0.15, 'records': 27513}, {'scenario': 'premium_0.02', 'start': 0.15, 'end': 0.1625, 'records': 12220}, {'scenario': 'premium_0.02', 'start': 0.1625, 'end': 0.175, 'records': 2903}, {'scenario': 'premium_0.02', 'start': 0.175, 'end': 0.1875, 'records': 188}, {'scenario': 'premium_0.02', 'start': 0.1875, 'end': 0.2, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2, 'end': 0.2125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.22499999999999998, 'end': 0.2375, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2375, 'end': 0.25, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.25, 'end': 0.26249999999999996, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.26249999999999996, 'end': 0.275, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.275, 'end': 0.2875, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2875, 'end': 0.3, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3, 'end': 0.3125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3125, 'end': 0.32499999999999996, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.32499999999999996, 'end': 0.3375, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3375, 'end': 0.35, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1125, 'end': 0.125, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.125, 'end': 0.1375, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1375, 'end': 0.15, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.15, 'end': 0.1625, 'records': 587}, {'scenario': 'premium_0.06', 'start': 0.1625, 'end': 0.175, 'records': 14629}, {'scenario': 'premium_0.06', 'start': 0.175, 'end': 0.1875, 'records': 21957}, {'scenario': 'premium_0.06', 'start': 0.1875, 'end': 0.2, 'records': 19133}, {'scenario': 'premium_0.06', 'start': 0.2, 'end': 0.2125, 'records': 13162}, {'scenario': 'premium_0.06', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 6874}, 

In [4]:
result.show()